# Tune `rf_k_rf`

RF top-K + Random Forest. Repeated stratified CV on the train split; 
writes [`data/processed/tuned/rf_k_rf.json`](../data/processed/tuned/rf_k_rf.json).

**Stage 1 (hyperparameters):** SelectFromModel `max_features` (K), classifier `max_depth`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

In [2]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "rf_k_rf"
spec = MODEL_SPECS[MODEL_ID]


In [3]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [4]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_mspc__select__max_features,classifier__max_depth
0,"[20, 30, 40, 50]",NaN
1,NaN,"[3, 4, 6, 8]"


In [5]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


rf_k_rf: 16 candidates x 25 folds = 400 fits


GridSearchCV 400 fits:   0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Fitting 25 folds for each of 16 candidates, totalling 400 fits


In [6]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,top_k,max_depth,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
15,50,8,48.328997,2.079712,0.516710,4.367647,4.425616,98.974359,0.628074,0.731134,0.040374,0.198982,0.046073
0,20,3,38.183258,5.067149,0.618167,35.941176,10.047616,87.692308,2.732370,0.720314,0.044346,0.198240,0.042723
8,40,3,40.596217,4.317673,0.594038,27.132353,9.293838,91.675214,2.417822,0.727797,0.039785,0.197582,0.046722
12,50,3,40.850867,3.981091,0.591491,25.426471,8.783414,92.871795,1.999781,0.728830,0.041424,0.197445,0.048144
1,20,4,39.992521,4.057434,0.600075,28.750000,8.543978,91.264957,2.284083,0.719258,0.043244,0.197332,0.046331
2,20,6,42.051156,4.261512,0.579488,20.735294,8.736705,95.162393,1.260329,0.713336,0.046752,0.197193,0.048665
11,40,8,47.362431,2.453936,0.526376,6.779412,5.029197,98.495726,0.617754,0.725049,0.042550,0.196691,0.051769
9,40,4,42.090121,3.782208,0.579099,20.794118,7.908484,95.025641,1.887632,0.725569,0.038484,0.196416,0.051556
13,50,4,43.888512,3.822914,0.561115,16.205882,7.884410,96.017094,1.433861,0.727949,0.039870,0.196218,0.050703
14,50,6,47.077866,2.553986,0.529221,7.485294,5.192947,98.358974,0.741967,0.728890,0.040228,0.195926,0.047696


In [7]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/25 [00:00<?, ?it/s]

Stage 2 best threshold: 0.20
  mean BER at threshold: 34.15%


,threshold,mean_ber_percent
0,0.20,34.152526
1,0.15,34.179110
2,0.25,34.665284
3,0.30,36.996732
4,0.35,40.005153
5,0.10,41.353570
6,0.40,43.000440
7,0.45,45.410194
8,0.50,48.328997
9,0.05,48.837607


In [8]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/rf_k_rf.json


{'preprocess__sensor_mspc__select__max_features': 50,
 'classifier__max_depth': 8}